In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import harpy as hp
import pandas as pd
import dask.dataframe as dd
import dask.array as da
from spatialdata import read_zarr

pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

In [ ]:
from ilastik.napari.object_classification import Object_Classifier
from napari_spatialdata import Interactive
from spatialdata.models import Labels2DModel

file_path = r"C:\Users\matti\Documents\WERK\STAGE\VIB\data\sdata_channels.zarr"
output_folder = r"C:\Users\matti\Documents\WERK\STAGE\VIB\output\object"

sdata = read_zarr(file_path)

oc = Object_Classifier(output_folder)

mask = sdata["masks_whole"].data
annotations = sdata["annotation"].data
image=[ sdata[ _image_name ].data for _image_name in [*sdata.images] ]
print(mask)
print(image)
print(annotations)
relabelled_masks = oc.object_classifier_workflow(mask, image, annotations)

se= Labels2DModel.parse(relabelled_masks)

sdata[  "predicted_labels" ] = se

Interactive(sdata)

c:\Users\matti\.conda\envs\ilastik_napari_182\Lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


dask.array<from-zarr, shape=(512, 512), dtype=int32, chunksize=(512, 512), chunktype=numpy.ndarray>
[dask.array<from-zarr, shape=(1, 512, 512), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>, dask.array<from-zarr, shape=(1, 512, 512), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>, dask.array<from-zarr, shape=(1, 512, 512), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>, dask.array<from-zarr, shape=(1, 512, 512), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>, dask.array<from-zarr, shape=(1, 512, 512), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>, dask.array<from-zarr, shape=(1, 512, 512), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>, dask.array<from-zarr, shape=(1, 512, 512), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>, dask.array<from-zarr, shape=(1, 512, 512), dtype=float32, chunksize=(1, 512, 512), chunktype=numpy.ndarray>, dask.array<from-zarr, shape

ValueError: Unsupported data type: <class 'napari._qt.qthreading.FunctionWorker'>.

In [8]:
from spatialdata import read_zarr

sdata=read_zarr(r"C:\Users\matti\Documents\WERK\STAGE\VIB\data\sdata_channels.zarr")
sdata

SpatialData object, with associated Zarr store: C:\Users\matti\Documents\WERK\STAGE\VIB\data\sdata_channels.zarr
├── Images
│     ├── 'channel_0': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_1': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_2': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_3': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_4': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_5': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_6': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_7': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_8': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_9': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_10': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_11': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_12': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_13': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_14': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_15': DataArray[cyx] (1, 512, 512)
│     ├── 'channel_16': DataArray[cyx]

see https://github.com/ilastik/ilastik/blob/54e17482cfe8c186a05b367450c4661792f7048c/ilastik/plugins_default/vigra_objfeats.py#L104 for all features ilastik extracts.

we probably want to support:

image + label:
- 'sum' 
- 'mean'
- 'var'
- 'kurtosis'
- 'skew'
- 'min'
- 'max'
- 'quantiles'

label:
- 'area'
- center_of_mass
- radii and axes

These are all implemented in `RasterAggregator`

In [9]:
sdata["masks_whole"].data

dask.array<from-zarr, shape=(512, 512), dtype=int32, chunksize=(512, 512), chunktype=numpy.ndarray>

In [10]:
import dask.array as da

mask=sdata[ "masks_whole" ].data[ None, ... ] # (z,y,x)

image=da.concatenate([ sdata[ _image_name ].data for _image_name in [*sdata.images] ])
image=image[ :, None, ... ] # ( c,z,y,x )

In [11]:
print(type(image))

<class 'dask.array.core.Array'>


In [12]:
from harpy.utils._aggregate import RasterAggregator

aggregator=RasterAggregator(mask_dask_array=mask, image_dask_array=image)

In [13]:
aggregator.aggregate_radii_and_axes( depth=100 ).head()

,0,1,2,3,4,5,6,7,8,9,10,11,cell_ID
0,2.439437,1.564841,0.0,0.0,-0.134019,0.990979,0.0,0.990979,0.134019,1.0,0.0,0.0,1
1,2.763416,1.501450,0.0,0.0,0.163683,0.986513,0.0,0.986513,-0.163683,1.0,0.0,0.0,2
2,2.975000,1.958264,0.0,0.0,-0.073496,0.997295,0.0,0.997295,0.073496,1.0,0.0,0.0,3
3,4.093265,2.402311,0.0,0.0,-0.092717,0.995693,0.0,0.995693,0.092717,1.0,0.0,0.0,4
4,5.219499,2.845574,0.0,0.0,0.225241,0.974303,0.0,0.974303,-0.225241,1.0,0.0,0.0,5


In [14]:
quantiles=aggregator.aggregate_quantiles(depth=100 ) # gives you a list of dataframes
quantiles[2].head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,cell_ID
0,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.007536,0.002808,0.000000,0.0,0.0,0.0,0.007817,0.000000,0.000000,0.004781,0.0,0.008071,0.009666,0.0,0.000003,0.0,1
1,0.000139,0.000000,0.00000,0.000000,0.000000,0.000427,0.015935,0.010562,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.003629,0.0,0.009900,0.018466,0.0,0.000000,0.0,2
2,0.000000,0.000000,0.00000,0.000602,0.000000,0.001172,0.004979,0.001632,0.004642,0.0,0.0,0.0,0.000000,0.000000,0.001588,0.000000,0.0,0.008341,0.001816,0.0,0.000000,0.0,3
3,0.000000,0.000000,0.00000,0.000000,0.000000,0.000383,0.010258,0.014653,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000488,0.000834,0.0,0.000000,0.0,4
4,0.000109,0.000233,0.00068,0.000000,0.000875,0.000387,0.006813,0.003731,0.000544,0.0,0.0,0.0,0.000000,0.000207,0.001023,0.001585,0.0,0.007116,0.009824,0.0,0.000000,0.0,5


In [15]:
dfs=aggregator.aggregate_stats( stats_funcs=("sum", "mean", "count", "var", "kurtosis"))

2025-03-06 14:52:03,852 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:04,073 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:04,460 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:05,144 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:05,785 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:06,019 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:06,263 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.


In [16]:
dfs[0] #-> sum, for each object, and each channel in image

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,cell_ID
0,129.162796,290.436920,58.454136,37.467636,115.047249,78.282028,40.574146,35.623764,75.485596,7.253437,14.956194,5.289702,210.020340,889.913452,326.641205,57.360939,17.141264,184.038956,308.747864,33.388474,54.782925,76.403786,0
1,0.014940,0.003414,0.010540,0.017549,0.011727,0.067361,0.583519,0.181400,0.021927,0.008699,0.015586,0.043255,0.484874,0.000000,0.091760,0.310179,0.000000,0.424088,0.564389,0.007468,0.032380,0.001899,1
2,0.285948,0.072576,0.023367,0.007146,0.003052,0.056604,1.120340,0.818089,0.157000,0.018974,0.021274,0.008716,0.026174,0.000000,0.055243,0.284302,0.004335,0.588799,0.887311,0.015834,0.004409,0.009961,2
3,0.065575,0.073491,0.041263,0.144170,0.056377,0.156341,0.712371,0.289304,0.403106,0.014571,0.058852,0.000035,0.024887,0.034710,0.142574,0.073501,0.003179,0.812896,0.295945,0.000035,0.011972,0.012868,3
4,0.033986,0.084177,0.077157,0.020228,0.054109,0.265152,2.180660,3.605335,0.071672,0.037721,0.041615,0.003794,0.036261,0.088938,0.087001,0.028337,0.003642,0.415661,0.320467,0.171777,0.015373,0.038468,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
670,0.087337,0.086334,0.232418,0.039902,0.477862,0.370914,4.790025,1.579594,0.018814,0.009748,0.078519,0.017383,0.179962,0.021400,0.472770,0.897789,0.030291,1.133461,2.925820,0.031743,0.457973,0.071657,670
671,0.037531,0.044381,0.108187,0.093129,0.119578,0.263704,2.059355,0.784729,0.034009,0.007169,0.148876,0.059225,0.137630,0.010464,0.334740,0.961098,0.013570,1.341750,2.560774,0.024349,0.702125,0.084365,671
672,0.122221,0.168703,0.103212,0.063834,0.071900,0.330067,13.462125,4.343424,0.234911,0.050640,0.093749,0.034504,3.031346,0.028846,1.469110,0.446479,0.013306,0.914507,1.742331,0.037230,0.360883,0.089279,672
673,0.018110,0.026148,0.053925,0.007099,0.069073,0.139027,0.921738,0.590694,0.005892,0.004721,0.007032,0.016624,0.016610,2.223094,0.376662,0.210060,0.011343,0.556679,0.466498,0.000000,0.018102,0.023747,673


In [17]:
sdata["annotation"].data

dask.array<from-zarr, shape=(512, 512), dtype=uint32, chunksize=(512, 512), chunktype=numpy.ndarray>

In [18]:
from ilastik.napari.utils import get_annotation

annotated_cells_id, annotation=get_annotation( array_1=sdata["annotation"].data, array_2=sdata["masks_whole"].data)

print(annotated_cells_id)
print(annotation)

[ 55  69  79  81  89  97  98 117 122 129 140 143 163 164 174 186 199 217
 218 473]
[ 2  3  1  1 10 10  1  1  2  1  2  2  2  2 10 10 10  3  2  3]


In [19]:
# for simplicity first try implementing object classification only using mean intensity
features=aggregator.aggregate_stats( stats_funcs=( "mean"  ) )# retuns a list of dataframes, take the first on (mean intensity)
print(len(features))
features[1][ [ 0, 1, "cell_ID" ] ] # only take mean, and only the first two channels
features=features[0][[ 0, 1, "cell_ID" ]]
features=features[  features[ "cell_ID" ]!=0 ] # remove features for background

4


In [20]:
def featuer_extractor(mask, image, stats):

    aggregator=RasterAggregator( mask_dask_array=mask, image_dask_array=image)

    features=aggregator.aggregate_stats(stats_funcs=stats)
    for index in range(len(stats)):
        prefix = stats[index]+"_"
        feature = features[index]
        feature.set_index("cell_ID")
        feature.columns = [f"{prefix}{c}" if f"{c}".isdigit() else c for c in feature.columns]

    res = dd.concat(features, axis=1)
    res = res.drop("cell_ID", axis=1)
    res = res.loc[res.index!=0]
    res["cell_ID"] = res.index

    return res

features = featuer_extractor(mask, image, stats=["sum", "mean", "count", "var", "kurtosis", "skew"])

2025-03-06 14:52:12,916 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:12,917 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'skewness' with 0 for affected cells.
2025-03-06 14:52:13,217 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:13,218 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'skewness' with 0 for affected cells.
2025-03-06 14:52:13,813 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:13,814 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'skewness' with 0 for affected cells.
2025-03-06 14:52:14,781 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'kurtosis' with 0 for affected cells.
2025-03-06 14:52:14,782 - harpy.utils._aggregate - WARNING - Replacing NaN values in 'skewness' with 0 for affected cells.
2025-03-06 14:52

In [21]:
features

,sum_0,sum_1,sum_2,sum_3,sum_4,sum_5,sum_6,sum_7,sum_8,sum_9,sum_10,sum_11,sum_12,sum_13,sum_14,sum_15,sum_16,sum_17,sum_18,sum_19,sum_20,sum_21,mean_0,mean_1,mean_2,mean_3,mean_4,mean_5,mean_6,mean_7,mean_8,mean_9,mean_10,mean_11,mean_12,mean_13,mean_14,mean_15,mean_16,mean_17,mean_18,mean_19,mean_20,mean_21,count_0,count_1,count_2,count_3,count_4,count_5,count_6,count_7,count_8,count_9,count_10,count_11,count_12,count_13,count_14,count_15,count_16,count_17,count_18,count_19,count_20,count_21,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,var_10,var_11,var_12,var_13,var_14,var_15,var_16,var_17,var_18,var_19,var_20,var_21,kurtosis_0,kurtosis_1,kurtosis_2,kurtosis_3,kurtosis_4,kurtosis_5,kurtosis_6,kurtosis_7,kurtosis_8,kurtosis_9,kurtosis_10,kurtosis_11,kurtosis_12,kurtosis_13,kurtosis_14,kurtosis_15,kurtosis_16,kurtosis_17,kurtosis_18,kurtosis_19,kurtosis_20,kurtosis_21,skew_0,skew_1,skew_2,skew_3,skew_4,skew_5,skew_6,skew_7,skew_8,skew_9,skew_10,skew_11,skew_12,skew_13,skew_14,skew_15,skew_16,skew_17,skew_18,skew_19,skew_20,skew_21,cell_ID
npartitions=1,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64
674,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [22]:
X_train=features[ features[ "cell_ID" ].isin( annotated_cells_id )]  # train on these
# drop the cell_ID column
X_train=X_train.drop("cell_ID", axis=1)
X_train.head()

,sum_0,sum_1,sum_2,sum_3,sum_4,sum_5,sum_6,sum_7,sum_8,sum_9,sum_10,sum_11,sum_12,sum_13,sum_14,sum_15,sum_16,sum_17,sum_18,sum_19,sum_20,sum_21,mean_0,mean_1,mean_2,mean_3,mean_4,mean_5,mean_6,mean_7,mean_8,mean_9,mean_10,mean_11,mean_12,mean_13,mean_14,mean_15,mean_16,mean_17,mean_18,mean_19,mean_20,mean_21,count_0,count_1,count_2,count_3,count_4,count_5,count_6,count_7,count_8,count_9,count_10,count_11,count_12,count_13,count_14,count_15,count_16,count_17,count_18,count_19,count_20,count_21,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,var_10,var_11,var_12,var_13,var_14,var_15,var_16,var_17,var_18,var_19,var_20,var_21,kurtosis_0,kurtosis_1,kurtosis_2,kurtosis_3,kurtosis_4,kurtosis_5,kurtosis_6,kurtosis_7,kurtosis_8,kurtosis_9,kurtosis_10,kurtosis_11,kurtosis_12,kurtosis_13,kurtosis_14,kurtosis_15,kurtosis_16,kurtosis_17,kurtosis_18,kurtosis_19,kurtosis_20,kurtosis_21,skew_0,skew_1,skew_2,skew_3,skew_4,skew_5,skew_6,skew_7,skew_8,skew_9,skew_10,skew_11,skew_12,skew_13,skew_14,skew_15,skew_16,skew_17,skew_18,skew_19,skew_20,skew_21
55,0.228718,0.836515,0.526362,0.207347,0.274174,1.742685,39.181973,22.877665,0.881706,0.089211,0.231409,0.104956,2.428303,0.230119,3.936078,0.826876,0.057538,2.743350,3.608209,0.174918,0.063279,0.123991,0.000409,0.001496,0.000942,0.000371,0.000490,0.003118,0.070093,0.040926,0.001577,0.000160,0.000414,0.000188,0.004344,0.000412,0.007041,0.001479,0.000103,0.004908,0.006455,0.000313,0.000113,0.000222,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,559.0,7.568810e-07,0.000017,2.195497e-06,5.912901e-07,9.261352e-07,0.000031,0.002825,0.001072,0.000010,2.343076e-07,5.824180e-07,2.039418e-07,0.000056,0.000001,0.000056,0.000005,9.354526e-08,0.000024,0.000048,3.882643e-07,1.553998e-07,2.867378e-07,21.154263,46.943073,5.234713,6.307810,8.316725,17.294182,0.922372,-0.929961,16.548750,18.355221,6.590362,8.127043,4.306724,17.491135,2.133469,2.654531,17.047197,1.100599,1.252757,5.692662,19.667192,10.600013,3.785487,6.231253,2.141033,2.489266,2.648656,3.846467,0.902874,0.490797,3.594724,4.017816,2.425029,2.803707,2.228747,3.750151,1.515362,1.760993,3.805794,1.240342,1.330835,2.354429,4.245370,3.073070
69,0.417686,0.449832,0.624587,0.125082,0.547636,0.416896,18.379158,9.084500,0.296956,0.136456,0.314453,0.038127,0.256381,2.992926,2.454963,2.233407,0.063546,2.072038,4.367913,0.093731,0.728978,0.104924,0.001088,0.001171,0.001627,0.000326,0.001426,0.001086,0.047862,0.023658,0.000773,0.000355,0.000819,0.000099,0.000668,0.007794,0.006393,0.005816,0.000165,0.005396,0.011375,0.000244,0.001898,0.000273,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,384.0,2.181469e-06,0.000003,4.422836e-06,4.081013e-07,3.556886e-06,0.000002,0.001353,0.000350,0.000002,6.740054e-07,1.381230e-06,1.062617e-07,0.000001,0.000102,0.000067,0.000028,1.742266e-07,0.000028,0.000139,4.137829e-07,5.231476e-05,3.447530e-07,1.804292,7.386664,1.622014,3.764820,5.556325,1.632245,0.246380,-0.008612,5.811791,13.310824,4.362662,17.388201,5.361319,3.944472,4.320229,0.005120,10.437126,4.078598,5.092445,10.544908,36.628002,7.315724,1.531526,2.430232,1.474319,2.097184,2.229709,1.507807,0.773828,0.831857,2.246113,3.380428,1.920424,4.035480,2.265422,1.945655,2.062723,0.883928,3.156258,1.861459,2.015800,3.203696,5.766249,2.629453
79,1.925154,3.064036,0.641999,0.778558,1.098652,1.392840,37.631718,16.047195,1.652116,0.104168,0.383137,0.068080,15.782252,0.328841,2.410144,1.001401,0.088205,3.447579,6.753321,0.515157,1.527694,0.242264,0.002711,0.004316,0.000904,0.001097,0.001547,0.001962,0.053002,0.022602,0.002327,0.000147,0.000540,0.000096,0.022229,0.000463,0.003395,0.001410,0.000124,0.004856,0.009512,0.000726,0.002152,0.000341,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,710.0,3.223713e-05,0.000102,1.892176e-06,3.735893e-0

In [23]:
annotation

array([ 2,  3,  1,  1, 10, 10,  1,  1,  2,  1,  2,  2,  2,  2, 10, 10, 10,
        3,  2,  3])

In [24]:
annotated_cells_id

array([ 55,  69,  79,  81,  89,  97,  98, 117, 122, 129, 140, 143, 163,
       164, 174, 186, 199, 217, 218, 473])

In [25]:
X_train

,sum_0,sum_1,sum_2,sum_3,sum_4,sum_5,sum_6,sum_7,sum_8,sum_9,sum_10,sum_11,sum_12,sum_13,sum_14,sum_15,sum_16,sum_17,sum_18,sum_19,sum_20,sum_21,mean_0,mean_1,mean_2,mean_3,mean_4,mean_5,mean_6,mean_7,mean_8,mean_9,mean_10,mean_11,mean_12,mean_13,mean_14,mean_15,mean_16,mean_17,mean_18,mean_19,mean_20,mean_21,count_0,count_1,count_2,count_3,count_4,count_5,count_6,count_7,count_8,count_9,count_10,count_11,count_12,count_13,count_14,count_15,count_16,count_17,count_18,count_19,count_20,count_21,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,var_10,var_11,var_12,var_13,var_14,var_15,var_16,var_17,var_18,var_19,var_20,var_21,kurtosis_0,kurtosis_1,kurtosis_2,kurtosis_3,kurtosis_4,kurtosis_5,kurtosis_6,kurtosis_7,kurtosis_8,kurtosis_9,kurtosis_10,kurtosis_11,kurtosis_12,kurtosis_13,kurtosis_14,kurtosis_15,kurtosis_16,kurtosis_17,kurtosis_18,kurtosis_19,kurtosis_20,kurtosis_21,skew_0,skew_1,skew_2,skew_3,skew_4,skew_5,skew_6,skew_7,skew_8,skew_9,skew_10,skew_11,skew_12,skew_13,skew_14,skew_15,skew_16,skew_17,skew_18,skew_19,skew_20,skew_21
npartitions=1,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32
674,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [26]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, annotation)
y_pred = clf.predict(X_train)
y_pred

array([ 2,  3,  1,  1, 10, 10,  1,  1,  2,  1,  2,  2,  2,  2, 10, 10, 10,
        3,  2,  3])

In [27]:
# run on all data
y_pred_all=clf.predict( features.drop( [ "cell_ID" ], axis=1 ) )  # gives us a prediction for every cell
y_pred_all.shape

(674,)

In [28]:
y_pred_all[:10]  # this is a label for every mask

array([ 2,  2,  2, 10,  2, 10, 10,  2,  2,  2])

In [29]:
cell_ids=features[ "cell_ID" ].compute() # cell_IDs
cell_ids
cell_ids[ :10 ]

1      1
2      2
3      3
4      4
5      5
6      6
7      7
8      8
9      9
10    10
Name: cell_ID, dtype: int64

In [30]:
# now generate the relabeld mask efficiently
mask=sdata[ "masks_whole" ].data # relabel this

In [31]:
import numpy as np
import dask.array as da

# create the relabeld mask as a dask array

assert cell_ids.shape == y_pred_all.shape

max_id = cell_ids.max()
lookup = np.zeros(max_id + 1, dtype=y_pred_all.dtype)
lookup[cell_ids] = y_pred_all 
relabelled_masks = da.take(lookup, mask) # maps each cell_id to its new label

In [32]:
from spatialdata.models import Labels2DModel

se= Labels2DModel.parse(relabelled_masks, dims=("y", "x"))

sdata[  "predicted_labels" ] = se

sdata.write(
    r"C:\Users\matti\Documents\WERK\STAGE\VIB\output\object\object_sdata.zarr",
    overwrite=True,
)

sdata = read_zarr(sdata.path)

INFO     The SpatialData object is not self-contained (i.e. it contains some elements that are Dask-backed from    
         locations outside C:\Users\matti\Documents\WERK\STAGE\VIB\output\object\object_sdata.zarr). Please see the
         documentation of `is_self_contained()` to understand the implications of working with SpatialData objects 
         that are not self-contained.                                                                              
INFO     The Zarr backing store has been changed from                                                              
         C:\Users\matti\Documents\WERK\STAGE\VIB\data\sdata_channels.zarr the new file path:                       
         C:\Users\matti\Documents\WERK\STAGE\VIB\output\object\object_sdata.zarr                                   


c:\Users\matti\.conda\envs\ilastik_napari_182\Lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


In [33]:
from napari_spatialdata import Interactive

Interactive( sdata )

In [34]:
# dummy code to explain the working of np.take
import numpy as np

lookup = np.array([0, 10, 20, 30, 40])

mask = np.array([
    [3, 1, 2],
    [3, 4, 0]
])

result = np.take(lookup, mask)
result

array([[30, 10, 20],
       [30, 40,  0]])